# 🧪 Praktikum 06 – Advanced Web-Scraping & Embedding-Engineering
**Applied Generative AI – NLP | Sommersemester 2026**

> 🎯 **Lernziele:** Rekursive Scraping-Strategien · Token-basierte Chunking-Verfahren · Vektorraum-Analyse und Distanzmetriken

⏱️ **Dauer:** 90 Minuten

In [ ]:
import os, sys, subprocess, time, shutil, re
from importlib.util import find_spec
MODEL = "qwen3.5:0.8b"
EMBED_MODEL = "nomic-embed-text"
def install_if_missing(pkgs):
    to_install = [p for p in pkgs if find_spec(p.replace('beautifulsoup4', 'bs4').replace('scikit-learn', 'sklearn')) is None]
    if to_install:
        cmd = [sys.executable, "-m", "pip", "install"] if not shutil.which("uv") else ["uv", "pip", "install"]
        if not (sys.prefix != getattr(sys, "base_prefix", sys.prefix)): cmd.append("--break-system-packages")
        subprocess.check_call(cmd + to_install)
install_if_missing(["requests", "beautifulsoup4", "ollama", "numpy", "scikit-learn", "matplotlib", "pandas", "tiktoken"])
import requests
from bs4 import BeautifulSoup
import numpy as np
import ollama
import tiktoken
import matplotlib.pyplot as plt
def setup_ollama():
    try: requests.get("http://localhost:11434/api/tags", timeout=2)
    except: subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL); time.sleep(5)
    subprocess.check_call(["ollama", "pull", MODEL])
    subprocess.check_call(["ollama", "pull", EMBED_MODEL])
setup_ollama()
print("✅ Setup abgeschlossen.")

## Teil 1 – Strukturiertes Scraping & Cleaning ⏱️ 25 min
Wir extrahieren Content, Metadaten und bereinigen den Text.

In [ ]:
def advanced_scrape(url):
    try:
        res = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'}, timeout=10)
        soup = BeautifulSoup(res.text, 'html.parser')
        data = []
        for section in soup.find_all(['h2', 'h3']):
            title = section.get_text().strip()
            text = ""
            for node in section.next_siblings:
                if node.name in ['h2', 'h3']: break
                if node.name == 'p': text += node.get_text().strip() + " "
            if len(text) > 100: data.append({"title": title, "content": text})
        return data
    except: return []

raw_data = advanced_scrape("https://de.wikipedia.org/wiki/Künstliche_Intelligenz")
if not raw_data:
    print("⚠️ Nutze Fallback-Daten.")
    raw_data = [{"title": "KI-Info", "content": "Künstliche Intelligenz ist ein Teilgebiet der Informatik." * 20}]

print(f"Gefundene Sektionen: {len(raw_data)}")

## Teil 2 – Token-basiertes Chunking ⏱️ 30 min
Wir nutzen `tiktoken` (OpenAI Standard).

In [ ]:
def token_chunker(text, max_tokens=100, overlap=20):
    enc = tiktoken.get_encoding("cl100k_base")
    tokens = enc.encode(text)
    chunks = []
    for i in range(0, len(tokens), max_tokens - overlap):
        chunks.append(enc.decode(tokens[i : i + max_tokens]))
    return chunks

all_chunks = []
for sec in raw_data:
    for c in token_chunker(sec['content']): all_chunks.append({"source": sec['title'], "text": c})

print(f"Anzahl Chunks: {len(all_chunks)}")
print(f"Beispiel-Chunk:\n{all_chunks[0]['text'][:100]}...")

## Teil 3 – Vektorraum-Analyse ⏱️ 35 min
Visualisierung semantischer Cluster.

In [ ]:
subset = all_chunks[:20]
vectors = np.array([ollama.embeddings(model=EMBED_MODEL, prompt=c['text'])['embedding'] for c in subset])

from sklearn.metrics.pairwise import cosine_distances
dist_matrix = cosine_distances(vectors)

plt.imshow(dist_matrix, cmap='viridis')
plt.title("Ähnlichkeitsmatrix")
plt.show()

## 🧩 Aufgaben
1. Vergleiche `Fixed Size` vs `Sliding Window` Chunking.
2. Implementiere eine Top-3 Vektorsuche.
3. Warum sind Token-Grenzen besser als Zeichen-Grenzen?